# Metrics data

In [25]:
import os
import numpy as np
import pandas as pd


curr = '20240215'
folder = f'Metrics/{curr}'

# Create output directory if it doesn't exist
output_folder = f'preprocessed_data/metrics/{curr}'
os.makedirs(output_folder, exist_ok=True)

for root, dirs, files in os.walk(folder):
    # Filter files that start with 'pod'
    pod_files = [f for f in files if f.startswith('pod')]
    
    for filename in pod_files:
        # Construct the relative path
        path = os.path.join(root, filename)
        
        # Load the NPZ file
        #data = np.load(path, allow_pickle=True).item()['book_info']
        data = np.load(path, allow_pickle=True).item()['pod usage']
        
        # Extract the components
        pod_names = data['Pod_Name']  # list of strings
        time = data['time']  # list of timestamps
        sequence = data['Sequence']  # shape: len(time) x (len(pod_name) + 1)
        
        # Create DataFrame
        # First column: time converted to datetime
        df_dict = {'time': pd.to_datetime(time, unit='s')}
        
        # Add columns for each pod name with their sequence values
        for i, pod_name in enumerate(pod_names):
            df_dict[pod_name] = sequence[:, i]
        
        # Last column: total (last entry of sequence for each time)
        df_dict['total'] = sequence[:, -1]
        
        # Create the DataFrame
        df = pd.DataFrame(df_dict)
        
        # Generate output filename with .csv extension
        # Remove any existing extension and add .csv
        base_name = os.path.splitext(filename)[0]  # Gets filename without extension
        output_filename = f"{base_name}.csv"
        output_path = os.path.join(output_folder, output_filename)
        
        # Save as CSV
        df.to_csv(output_path, index=False)
        
        print(f"Processed: {filename} -> {output_filename}")


Processed: pod_level_data_pod_cpu_utilization_over_pod_limit.npy -> pod_level_data_pod_cpu_utilization_over_pod_limit.csv
Processed: pod_level_data_pod_memory_utilization_over_pod_limit.npy -> pod_level_data_pod_memory_utilization_over_pod_limit.csv
Processed: pod_level_data_pod_network_tx_bytes.npy -> pod_level_data_pod_network_tx_bytes.csv
Processed: pod_level_data_pod_cpu_limit.npy -> pod_level_data_pod_cpu_limit.csv
Processed: pod_level_data_pod_network_rx_bytes.npy -> pod_level_data_pod_network_rx_bytes.csv
Processed: pod_level_data_pod_memory_limit.npy -> pod_level_data_pod_memory_limit.csv
Processed: pod_level_data_pod_cpu_usage_total.npy -> pod_level_data_pod_cpu_usage_total.csv


# Log data

In [ ]:
import os
import pandas as pd
import re
from pathlib import Path


def preprocess_pod_logs(log_folder, curr='20231207', output_folder='preprocessed_data/logs'):
    """
    Preprocess pod logs by merging structured and template files
    """
    folder_path = f'{log_folder}/{curr}/log_data/pod_message'
    output_path = f'{output_folder}/{curr}/pods'
    os.makedirs(output_path, exist_ok=True)
    
    # Get all files in the folder
    all_files = os.listdir(folder_path)
    
    # Find all unique pod identifiers
    pod_ids = set()
    for filename in all_files:
        if '_messages_structured.csv' in filename:
            # Extract pod identifier (everything before _messages_structured.csv)
            pod_id = filename.replace('_messages_structured.csv', '')
            pod_ids.add(pod_id)
    
    print(f"Found {len(pod_ids)} pods to process\n")
    
    # Process each pod
    for pod_id in sorted(pod_ids):
        structured_file = os.path.join(folder_path, f'{pod_id}_messages_structured.csv')
        templates_file = os.path.join(folder_path, f'{pod_id}_messages_templates.csv')
        
        # Check if both files exist
        if not os.path.exists(structured_file):
            print(f"Warning: Missing structured file for {pod_id}")
            continue
        if not os.path.exists(templates_file):
            print(f"Warning: Missing templates file for {pod_id}")
            continue
        
        try:
            # Load structured logs (actual log instances)
            structured_df = pd.read_csv(structured_file)
            
            # Load templates (event patterns)
            templates_df = pd.read_csv(templates_file)
            
            # Check if EventId/eventid column exists
            eventid_col_struct = find_eventid_column(structured_df)
            eventid_col_template = find_eventid_column(templates_df)
            
            if eventid_col_struct is None or eventid_col_template is None:
                print(f"Warning: EventId column not found for {pod_id}")
                continue
            
            # Standardize column names to 'eventid'
            if eventid_col_struct != 'eventid':
                structured_df.rename(columns={eventid_col_struct: 'eventid'}, inplace=True)
            if eventid_col_template != 'eventid':
                templates_df.rename(columns={eventid_col_template: 'eventid'}, inplace=True)
            
            # Remove duplicates from templates
            templates_df = templates_df.drop_duplicates(subset='eventid', keep='first')
            
            # Merge structured logs with templates on eventid
            merged_df = structured_df.merge(
                templates_df, 
                on='eventid', 
                how='left',
                suffixes=('', '_template')
            )
            
            # Convert timestamp if present
            timestamp_cols = [col for col in merged_df.columns if 'time' in col.lower()]
            for ts_col in timestamp_cols:
                try:
                    # Try different timestamp formats
                    merged_df[ts_col] = pd.to_datetime(merged_df[ts_col], errors='coerce')
                except:
                    pass
            
            # Sort by timestamp if available
            if timestamp_cols:
                merged_df = merged_df.sort_values(timestamp_cols[0]).reset_index(drop=True)
            
            # Save merged data
            output_file = os.path.join(output_path, f'{pod_id}_logs_merged.csv')
            merged_df.to_csv(output_file, index=False)
            
            print(f"✓ Processed {pod_id}: {len(structured_df)} logs → {output_file}")
            
        except Exception as e:
            print(f"✗ Error processing {pod_id}: {str(e)}")
            continue
    
    print(f"\nPreprocessing complete! Output saved to: {output_path}")
    return True


def find_eventid_column(df):
    """
    Find the EventId column (case-insensitive)
    """
    for col in df.columns:
        if col.lower() in ['eventid', 'event_id', 'eventtemplate']:
            return col
    return None


# Usage
preprocess_pod_logs('Log', curr='20231221', output_folder='preprocessed_data/logs')


Found 370 pods to process

✓ Processed adservice-7df8c84f69-nvnc4: 163085 logs → preprocessed_data/logs/20231221/pods/adservice-7df8c84f69-nvnc4_logs_merged.csv
✓ Processed adservice-7df8c84f69-zgsdx: 416971 logs → preprocessed_data/logs/20231221/pods/adservice-7df8c84f69-zgsdx_logs_merged.csv
✓ Processed aws-load-balancer-controller-69bd799c8c-489sn: 17 logs → preprocessed_data/logs/20231221/pods/aws-load-balancer-controller-69bd799c8c-489sn_logs_merged.csv
✓ Processed aws-load-balancer-controller-69bd799c8c-6p8nk: 17 logs → preprocessed_data/logs/20231221/pods/aws-load-balancer-controller-69bd799c8c-6p8nk_logs_merged.csv
✓ Processed aws-load-balancer-controller-69bd799c8c-8mmpq: 57 logs → preprocessed_data/logs/20231221/pods/aws-load-balancer-controller-69bd799c8c-8mmpq_logs_merged.csv
✓ Processed aws-load-balancer-controller-69bd799c8c-f8qn9: 35 logs → preprocessed_data/logs/20231221/pods/aws-load-balancer-controller-69bd799c8c-f8qn9_logs_merged.csv
✓ Processed aws-load-balancer-con

In [16]:
import os
import numpy as np
import pandas as pd


curr = '20240215/log_data'
folder = f'Log/{curr}'

# Create output directory if it doesn't exist
output_folder = f'preprocessed_data/logs/{curr}'
os.makedirs(output_folder, exist_ok=True)

for root, dirs, files in os.walk(folder):
    # Filter files that start with 'pod'
    pod_files = [f for f in files if f.startswith('pod')]
    
    for filename in pod_files:
        # Construct the relative path
        path = os.path.join(root, filename)
        
        # Load the NPZ file
        # Check if it's a golden file
        # if 'golden' in filename.lower():
        #     # For golden files, only use .item()
        #     data = np.load(path, allow_pickle=True).item()
        # else:
        #     # For non-golden files, access the specific key
        #     data = np.load(path, allow_pickle=True).item()['ratings.book-info.svc.cluster.local:9080/*']
        data = np.load(path, allow_pickle=True).item()['book_info']
        # Extract the components
        node_names = data['Node_Name']  # list of strings (98 nodes)
        time = data['time']  # list of timestamps (5535 time points)
        sequence = data['Sequence']  # shape: (98, 5535, 1)
        
        # Reshape sequence from 3D to 2D by squeezing the last dimension
        if sequence.ndim == 3:
            sequence = sequence.squeeze(-1)  # (98, 5535, 1) -> (98, 5535)
        
        # TRANSPOSE: We need (time_points, nodes) not (nodes, time_points)
        # After transpose: (5535, 98)
        sequence = sequence.T
        
        # Verify the shape
        print(f"Processing {filename}")
        print(f"  File type: {'Golden' if 'golden' in filename.lower() else 'Regular'}")
        print(f"  Time length: {len(time)}")
        print(f"  Nodes: {len(node_names)}")
        print(f"  Sequence shape after transpose: {sequence.shape}")
        
        # Create DataFrame
        # First column: time converted to datetime
        df_dict = {'time': pd.to_datetime(time, unit='s')}
        
        # Add columns for each node name with their sequence values
        for i, node_name in enumerate(node_names):
            df_dict[node_name] = sequence[:, i]
        
        # Last column: total (last entry of sequence for each time)
        df_dict['total'] = sequence[:, -1]
        
        # Create the DataFrame
        df = pd.DataFrame(df_dict)
        
        print(f"  DataFrame shape: {df.shape}")
        
        # Generate output filename with .csv extension
        base_name = os.path.splitext(filename)[0]
        output_filename = f"{base_name}.csv"
        output_path = os.path.join(output_folder, output_filename)
        
        # Save as CSV
        df.to_csv(output_path, index=False)
        
        print(f"  ✓ Saved: {output_filename}\n")


Processing pod_level_log_frequency.npy
  File type: Regular
  Time length: 5748
  Nodes: 749
  Sequence shape after transpose: (5748, 749)
  DataFrame shape: (5748, 751)
  ✓ Saved: pod_level_log_frequency.csv

Processing pod_level_log_golden_signal.npy
  File type: Golden
  Time length: 5745
  Nodes: 15
  Sequence shape after transpose: (5745, 15)
  DataFrame shape: (5745, 17)
  ✓ Saved: pod_level_log_golden_signal.csv



In [15]:
data = np.load('Log/20240215/log_data/pod_level_log_golden_signal.npy', allow_pickle=True).item()
data.keys()

dict_keys(['book_info'])